In [0]:
# importing libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score , mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor


In [0]:
# load the dataframe and transform  to pandas 
df=spark.table('workspace.silver.silver_multimodal').toPandas()

In [0]:
# displaying the dataset
df.head(5)

In [0]:
# filter the dataset 
df=df[df['site_id']=='10004']
df.count()

In [0]:
# displaying the dataset
df.head(5)

In [0]:
#  dataset shape 
df.shape

In [0]:
# data type
df.dtypes

In [0]:
# creat data column from timestamp 
df['date']=df['timestamp'].dt.date
df.head(5)

In [0]:
# transport mode
df['transport_mode'].unique()

In [0]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

df_2 = df.pivot_table(
    index='date',
    columns='transport_mode',
    values='timestamp',
    aggfunc='count',
    fill_value=0
)

# Add daily total
df_2['total'] = df_2.sum(axis=1)

# Clean column names and reset index
df_2.columns.name = None
df_2 = df_2.reset_index()

# Sort chronologically and create index starting at 0
df_2 = df_2.sort_values('date').reset_index(drop=True)
df_2.insert(0, 'index', range(len(df_2)))

df_2.tail(20)

In [0]:
# Showing a date interval
df_port = df_2[
    (df_2['date'] < '2026-03-03') &
    (df_2['date'] > '2026-01-01')
]

df_port

In [0]:
# dropping unasseary column 
df_2=df_2.drop(columns='index')
df_2

In [0]:
# data description
df_2.describe()

In [0]:
# display dataframe
df_2

In [0]:
# Strip any hidden spaces from the column names in your existing DataFrame
df_2.columns = df_2.columns.str.strip()

# list vehiculs col
vehicle_cols = ['Bicycles', 'Buses and Coaches', 'Heavy Vehicles (> 3.5t)', 
                'Light Vehicles (< 3.5t)', 'Motorized Two-Wheelers', 'Scooters']

totals = df_2[vehicle_cols].sum()
averages = df_2[vehicle_cols].mean()

In [0]:
# Graph : Line Chart (Total Traffic Over Time)
plt.figure(figsize=(12, 4))
plt.plot(df_2['date'], df_2['total'], color='purple')
plt.title('1. Total Daily Traffic Over Time')
plt.xlabel('Date')
plt.ylabel('Total Vehicles')
plt.tight_layout()
plt.show()

In [0]:
# Graph : Vertical Bar Chart (Total Volume by Vehicle Type)
plt.figure(figsize=(10, 5))
plt.bar(totals.index, totals.values, color='skyblue', edgecolor='black')
plt.title('2. Total Volume by Vehicle Type')
plt.ylabel('Total Count')
plt.xticks(rotation=15) 
plt.tight_layout()
plt.show()

In [0]:
# Graph : Pie Chart (Proportion of Vehicle Types)
plt.figure(figsize=(7, 7))
plt.pie(totals.values, labels=totals.index, autopct='%1.1f%%', startangle=90)
plt.title('3. Proportion of Traffic by Vehicle Type')
plt.tight_layout()
plt.show()

In [0]:
# Graph : Histogram (Distribution of Total Daily Traffic)
plt.figure(figsize=(8, 5))
plt.hist(df_2['total'], bins=30, color='green', alpha=0.7, edgecolor='black')
plt.title('4. Distribution of Total Daily Traffic')
plt.xlabel('Total Vehicles per Day')
plt.ylabel('Frequency (Days)')
plt.tight_layout()
plt.show()

In [0]:
# Graph : Scatter Plot (Light Vehicles vs. Motorized Two-Wheelers)
plt.figure(figsize=(8, 5))
plt.scatter(df_2['Light Vehicles (< 3.5t)'], df_2['Motorized Two-Wheelers'], alpha=0.5, color='teal')
plt.title('6. Light Vehicles vs. Motorized Two-Wheelers')
plt.xlabel('Light Vehicles')
plt.ylabel('Motorized Two-Wheelers')
plt.tight_layout()
plt.show()

In [0]:
# Graph : Stacked Area Chart (Bicycles and Scooters Over Time)
plt.figure(figsize=(12, 5))
plt.stackplot(df_2['date'], df_2['Bicycles'], df_2['Scooters'], labels=['Bicycles', 'Scooters'], colors=['#ff9999','#66b3ff'])
plt.title('7. Area Chart: Bicycles and Scooters Volume')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [0]:
# GRAPH 2: Scatter Matrix (Pairplot)

pd.plotting.scatter_matrix(df_2, figsize=(12, 10), diagonal='kde', alpha=0.5, color='darkcyan')
plt.suptitle('2. Scatter Matrix (Feature Relationships & Density)', y=1.02)
plt.show()

In [0]:
# GRAPH 3: Boxplots for Outlier Detection

plt.figure(figsize=(10, 6))
df_2.plot.box(vert=False, figsize=(10, 6), patch_artist=True)
plt.title('3. Outlier Detection Across Features')
plt.xlabel('Values')
plt.show()

In [0]:
# GRAPH : Feature Distributions (Histograms)
df_2.hist(figsize=(10, 8), bins=30, color='steelblue', edgecolor='black')
plt.suptitle('4. Feature Distributions (Check for Skewness)', y=1.02)
plt.tight_layout()
plt.show()

Machine learning


In [0]:
# 1. Feature Engineering (Convert dates into numbers the ML model can understand)
df_2['date'] = pd.to_datetime(df_2['date'])

df_ml = pd.DataFrame()
df_ml['date'] = df_2['date']
df_ml['total'] = df_2['total']

In [0]:
# Extract time-based features
df_ml['year'] = df_ml['date'].dt.year
df_ml['month'] = df_ml['date'].dt.month
df_ml['day_of_week'] = df_ml['date'].dt.dayofweek
df_ml['day_of_year'] = df_ml['date'].dt.dayofyear

In [0]:
# Define inputs (X) and what we want to predict (y)
X = df_ml[['year', 'month', 'day_of_week', 'day_of_year']]
y = df_ml['total']

In [0]:
# 2. Train/Test Split (Keep chronological order using shuffle=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [0]:
# 3. Train the Machine Learning Model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [0]:
# 4. Calculate the Accuracy Score on the Test Set
test_predictions = model.predict(X_test)
accuracy_r2 = r2_score(y_test, test_predictions)
mae = mean_absolute_error(y_test, test_predictions)

print(f"--- Model Performance on Historical Data ---")
print(f"Accuracy Score (R²): {accuracy_r2:.2f} ")
print(f"Mean Absolute Error: Misses by ~{mae:.0f} vehicles per day on average\n")

In [0]:
# 5. Generate Future Dates (Every day from 2027 to 2030)
future_dates = pd.date_range(start='2027-01-01', end='2027-12-31', freq='D')
df_future = pd.DataFrame({'date': future_dates})

In [0]:
# Extract the exact same features for the future dates
df_future['year'] = df_future['date'].dt.year
df_future['month'] = df_future['date'].dt.month
df_future['day_of_week'] = df_future['date'].dt.dayofweek
df_future['day_of_year'] = df_future['date'].dt.dayofyear

In [0]:
# 6. Predict Future Totals
X_future = df_future[['year', 'month', 'day_of_week', 'day_of_year']]
df_future['projected_total'] = model.predict(X_future)

In [0]:
# 7. Clean up the Final DataFrame
df_final_projections = df_future[['date', 'projected_total']].copy()
df_final_projections['projected_total'] = df_final_projections['projected_total'].round().astype(int)

In [0]:
# Display the resulting dataframe in Databricks
display(df_final_projections)

# model 2 

In [0]:
# %pip install statsmodels

In [0]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing













In [0]:
# 1. Prepare data strictly as a chronological time series
df_ts = df_2[['date', 'total']].copy()
df_ts['date'] = pd.to_datetime(df_ts['date'])
df_ts = df_ts.set_index('date').sort_index()

In [0]:
# 2. Fix missing days: Time series math breaks if a single day is missing.
# This forces a daily frequency ('D') and fills gaps by interpolating the line.
df_ts = df_ts.resample('D').interpolate(method='linear')

In [0]:
# 3. Train the Time Series Model
# 'trend' carries recent upward/downward momentum forward.
# 'seasonal_periods=7' forces the model to mimic the 7-day weekly traffic cycle.
model = ExponentialSmoothing(
    df_ts['total'], 
    trend='add', 
    seasonal='add', 
    seasonal_periods=7,
    initialization_method="estimated"
).fit()


In [0]:
# 4. Predict exactly for 2027
# We predict from Jan 1 to Dec 31
start_date = pd.to_datetime('2027-01-01')
end_date = pd.to_datetime('2027-12-31')
predictions = model.predict(start=start_date, end=end_date)

In [0]:
# 5. Clean up Final DataFrame
df_final_projections_2 = predictions.reset_index()
df_final_projections_2.columns = ['date', 'projected_total']
df_final_projections_2['projected_total'] = df_final_projections_2['projected_total'].round().astype(int)


In [0]:
# 6. Visualize the new model
plt.figure(figsize=(12, 4))
plt.plot(df_ts.index[-150:], df_ts['total'][-150:], label='Historical (Last 150 Days)', color='purple')
plt.plot(df_final_projections_2['date'], df_final_projections_2['projected_total'], label='2027 Projection', color='green')
plt.title('True Time-Series Projection for 2027 (Holt-Winters)')
plt.legend()
plt.show()

In [0]:
# Display the resulting dataframe in Databricks
display(df_final_projections_2)